# FEWS Agent Tools Validation

This notebook validates the tools available in the FEWS Configuration Agent without using the LLM. It directly tests each tool's functionality to ensure they work as expected.

In [1]:
import sys
from pathlib import Path
from IPython.display import display, Markdown

# Add parent directory to path
sys.path.append(Path().absolute().parent.as_posix())

# Import required modules
from talkwithfews.tools import (
    execute_query,
    explain_query,
    search_workflows,
    get_workflow_details,
    search_timeseries,
    get_timeseries_details,
    search_transformations,
    get_transformation_details,
    analyze_upstream_dependencies,
    analyze_downstream_dependencies,
    find_workflow_call_hierarchy,
    get_database_schema,
    analyze_dead_points
)

from talkwithfews.visualization import (
    get_endpoint_to_root_path,
    render_path_as_flow,
    render_path_as_table
)

from talkwithfews.models import (
    Deps,
    SQLQueryInput,
    WorkflowSearchInput,
    TimeSeriesSearchInput,
    TransformationSearchInput,
    DependencyAnalysisInput,
    ToolCallTracker,
    EndpointPathRequest,
    RenderPathInput,
    RenderPathTableInput
)

from talkwithfews.utils import connect_to_database
from talkwithfews.logging_utils import setup_logging
from pydantic_ai import RunContext

# Set logging level to ERROR at the start of the notebook
setup_logging(log_level="ERROR")

2025-03-27 23:54:12.869 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [2]:
# Set up database connection
db_path = '../db/fews_config.db3'
db_connection = connect_to_database(db_path)

tracker = ToolCallTracker()

# Create dependencies object
deps = Deps(
    db_path=db_path, 
    db_connection=db_connection,
    tracker=tracker
)

# Create run context with mock values for required parameters
ctx = RunContext(
    deps=deps,
    model="mock-model",  # Mock model name since we're not using LLM
    usage={              # Mock usage statistics
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0
    },
    prompt="mock-prompt" # Mock prompt since we're not using LLM
)

## 1. Database Schema Validation

In [ ]:
# Test get_database_schema
schema = await get_database_schema(ctx)
print("Database Schema:")
print(f"Number of tables: {len(schema['tables'])}")
print(f"Number of relationships: {len(schema['relationships'])}")

print("\nRaw Schema Structure:")
print("Tables:")
for table_name, columns in schema['tables'].items():
    print(f"- {table_name}:")
    print(f"  Columns: {', '.join(columns)}")

print("\nRelationships:")
for rel in schema['relationships']:
    print(f"{rel}")

print("\nSample Data:")
for table_name, samples in schema['sample_data'].items():
    print(f"\n{table_name}:")
    for sample in samples[:2]:  # Show first 2 samples for brevity
        print(f"  {sample}")
        
print("\nTables:")
for table_name, columns in schema['tables'].items():
    print(f"- {table_name}: {len(columns)} columns")

## 2. Query Execution Validation

In [ ]:
# Test execute_query with a simple query
simple_query = "SELECT COUNT(*) as count FROM trafos"
result = await execute_query(ctx, SQLQueryInput(query=simple_query))
print("Number of transformations:")
print(result.results[0]['count'])

# Test explain_query
explanation = await explain_query(ctx, SQLQueryInput(query=simple_query))
print("\nQuery explanation:")
print(explanation)

## 3. Workflow Tools Validation

In [ ]:
# Test search_workflows
workflow_search = await search_workflows(ctx, WorkflowSearchInput(name_pattern="water"))
print("Workflows containing 'water':")
for workflow in workflow_search.results:
    print(f"- {workflow['wf_id']} (ID: {workflow['id']}) - {workflow['desc']}")

# Test get_workflow_details if we found any workflows
if workflow_search.results:
    workflow_id = workflow_search.results[0]['id']
    details = await get_workflow_details(ctx, {'workflow_id': workflow_id})
    print("\nFirst workflow details:")
    print(f"ID: {details.results[0]['id']}")
    print(f"Workflow ID: {details.results[0]['wf_id']}")
    print(f"Description: {details.results[0]['desc']}")

## 4. Time Series Tools Validation

In [ ]:
# Test search_timeseries
ts_search = await search_timeseries(ctx, TimeSeriesSearchInput(
    parameter_id="water",  # Search in parameter IDs
    qualifier_pattern=None,
    module_instance=None
))
print("Time series containing 'water':")
for ts in ts_search.results:
    print(f"- {ts['parid']} (ID: {ts['id']}) - Qualifiers: {ts['qualifiers']}")

# Test get_timeseries_details if we found any time series
if ts_search.results:
    ts_id = ts_search.results[0]['id']
    details = await get_timeseries_details(ctx, {'timeseries_id': ts_id})
    print("\nFirst time series details:")
    print(f"Parameter ID: {details.results[0]['parid']}")
    print(f"Qualifiers: {details.results[0]['qualifiers']}")
    print(f"Type: {details.results[0]['tstype']}")

## 5. Transformation Tools Validation

In [ ]:
# Test search_transformations
trans_search = await search_transformations(ctx, TransformationSearchInput(description_pattern="water"))
print("Transformations containing 'water':")
for trans in trans_search.results:
    print(f"- {trans['desc']} (ID: {trans['id']})")

# Test get_transformation_details if we found any transformations
if trans_search.results:
    trans_id = trans_search.results[0]['id']
    details = await get_transformation_details(ctx, {'transformation_id': trans_id})
    print("\nFirst transformation details:")
    print(f"Description: {details.results[0]['desc']}")
    print(f"Module Instance: {details.results[0]['modins']}")

## 6. Dependency Analysis Tools Validation

In [ ]:
# Test analyze_upstream_dependencies
if ts_search.results:
    ts_id = ts_search.results[0]['id']
    upstream = await analyze_upstream_dependencies(ctx, DependencyAnalysisInput(
        entity_id=ts_id,
        entity_type='timeseries'
    ))
    print("Upstream dependencies:")
    print(f"Number of producers: {len(upstream.results)}")
    if upstream.results:
        print("First producer:")
        print(f"- Type: {upstream.results[0]['type'] if 'type' in upstream.results[0] else 'N/A'}")
        print(f"- ID: {upstream.results[0]['id']}")

    # Test analyze_downstream_dependencies
    downstream = await analyze_downstream_dependencies(ctx, DependencyAnalysisInput(
        entity_id=ts_id,
        entity_type='timeseries'
    ))
    print("\nDownstream dependencies:")
    print(f"Number of consumers: {len(downstream.results)}")
    if downstream.results:
        print("First consumer:")
        print(f"- Type: {downstream.results[0]['type'] if 'type' in downstream.results[0] else 'N/A'}")
        print(f"- ID: {downstream.results[0]['id']}")

## 7. Workflow Call Hierarchy Validation

In [ ]:
# Test find_workflow_call_hierarchy
if workflow_search.results:
    root_workflow_id = workflow_search.results[0]['id']
    hierarchy = await find_workflow_call_hierarchy(ctx, {'root_workflow_id': root_workflow_id})  # Changed key name
    print("Workflow call hierarchy:")
    print(f"Number of levels: {len(hierarchy.results)}")
    
    # Group results by level
    levels = {}
    for item in hierarchy.results:
        level = item['level']
        if level not in levels:
            levels[level] = []
        levels[level].append(item)
    
    # Print each level
    for level, workflows in sorted(levels.items()):
        print(f"\nLevel {level}:")
        print(f"Number of workflows: {len(workflows)}")
        if workflows:
            print(f"First workflow: {workflows[0]['wf_id']}")

## 8. Dead Points Analysis Validation

In [ ]:
# Test analyze_dead_points
dead_points = await analyze_dead_points(ctx)
print("Dead points analysis:")
print(f"Total dead points found: {dead_points['count']}")
if dead_points['dead_points']:
    print("\nFirst dead point:")
    first_dp = dead_points['dead_points'][0]
    print(f"- ID: {first_dp['id']}")
    print(f"- Parameter ID: {first_dp['parid']}")
    print(f"- Module Instance: {first_dp['modins']}")

## 9. Endpoint to rooth path

In [3]:
# Test with timeseries set ID 336
path_data = await get_endpoint_to_root_path(ctx, EndpointPathRequest(
    entity_id=336,
    depth=5  # Limit the depth to 5 levels
))

print("Path data for timeseries set ID 336:")
print(f"Number of entries: {len(path_data.data)}")

# Print the path structure in a hierarchical format
print("\nPath structure from endpoint to root:")
for entry in path_data.data:
    # Print timeseries object
    print("\nTimeseries Object:")
    print(f"  ID: {entry['tss_id']}")
    print(f"  Module: {entry['modins']}")
    print(f"  Value Type: {entry['vtype']}")
    print(f"  Parameter: {entry['parid']}")
    if entry['qualifiers']:
        print(f"  Qualifiers: {entry['qualifiers']}")
    print(f"  Type: {entry['tstype']}")
    print(f"  Time Step: {entry['timess']}")
    
    # Print transformation object if present
    if entry['trafo_desc'] and entry['wf_desc']:
        print("\nTransformation Object:")
        print(f"  ID: {entry['trafo_id']}")
        print(f"  Description: {entry['trafo_desc']}")
        print(f"  Workflow: {entry['wf_id']}")
    
    print("-" * 50)  # Separator between entries

Path data for timeseries set ID 336:
Number of entries: 24

Path structure from endpoint to root:

Timeseries Object:
  ID: 333
  Module: Hydronet_wic
  Value Type: scalar
  Parameter: WNS3380.volume
  Type: external historical
  Time Step: 1day

Transformation Object:
  ID: 46
  Description: Sluitpost.tussenstap
  Workflow: Aggregaties
--------------------------------------------------

Timeseries Object:
  ID: 333
  Module: Hydronet_wic
  Value Type: scalar
  Parameter: WNS3380.volume
  Type: external historical
  Time Step: 1day

Transformation Object:
  ID: 46
  Description: Sluitpost.tussenstap
  Workflow: Aggregaties
--------------------------------------------------

Timeseries Object:
  ID: 333
  Module: Hydronet_wic
  Value Type: scalar
  Parameter: WNS3380.volume
  Type: external historical
  Time Step: 1day

Transformation Object:
  ID: 46
  Description: Sluitpost.tussenstap
  Workflow: Aggregaties
--------------------------------------------------

Timeseries Object:
  ID: 

## 10. Test render as strealit-flow component

In [5]:
# Test rendering the path as a flow diagram
flow_output = await render_path_as_flow(ctx, RenderPathInput(path_data=path_data))

# Print the flow state information
print("Flow diagram state:")
print(f"Number of nodes: {len(flow_output.state['nodes'])}")
print(f"Number of edges: {len(flow_output.state['edges'])}")

# Print the node and edge details
print("\nNode details:")
for node in flow_output.state['nodes']:
    print(f"\nNode ID: {node['id']}")
    print(f"Content: {node['data']['content']}")

print("\nEdge details:")
for edge in flow_output.state['edges']:
    print(f"\nEdge ID: {edge['id']}")
    print(f"Source: {edge['source']}")
    print(f"Target: {edge['target']}")

Flow diagram state:
Number of nodes: 24
Number of edges: 24

Node details:

Node ID: ff2354a2-10e6-4260-a7bc-8df112adf823
Content: Timeseries Set:
Module: GenerateProcesOverzichtTimeseries
Value Type: scalar
Parameter: WNS2366
Type: external historical
Time Step: 1day

Node ID: 18446f5f-a20e-40a8-9eeb-baaf6defc6bd
Content: Timeseries Set:
Module: GenerateProcesOverzichtTimeseries
Value Type: scalar
Parameter: WNS2366
Type: external historical
Time Step: 1day

Node ID: 0dbe33f8-06ae-4bd9-a890-89555795cf98
Content: Timeseries Set:
Module: GenerateProcesOverzichtTimeseries
Value Type: scalar
Parameter: WNS2366
Type: external historical
Time Step: 1day

Node ID: d4b8efd5-04fc-4757-a3a4-133608c7aace
Content: Timeseries Set:
Module: GenerateProcesOverzichtTimeseries
Value Type: scalar
Parameter: WNS2366
Type: external historical
Time Step: 1day

Node ID: 79a7122c-e2ff-4b4b-9b29-686d66e8595c
Content: Timeseries Set:
Module: GenerateProcesOverzichtTimeseries
Value Type: scalar
Parameter: WNS23

## Cleanup

In [12]:
# Close database connection
# db_connection.close()